# Classification

In [ ]:
import lime
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from lime.lime_tabular import LimeTabularExplainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns

# Titanic Dataset

The Titanic dataset is a famous dataset used in data science and machine learning, providing details about the passengers on the Titanic, including whether they survived the ship's sinking.

## Key Characteristics:

- **Number of Entries**: Over 800 for the training dataset.
- **Features**: Includes various passenger attributes like age, sex, ticket class, and fare.
- **Target Variable**: Survival status (survived or did not survive).

## Features Include:

- **PassengerId**: A unique identifier for each passenger.
- **Survived**: The target variable; 0 = No, 1 = Yes.
- **Pclass**: Ticket class; 1 = 1st, 2 = 2nd, 3 = 3rd.
- **Name**: The passenger's name.
- **Sex**: The passenger's sex.
- **Age**: The passenger's age.
- **SibSp**: The number of siblings/spouses aboard.
- **Parch**: The number of parents/children aboard.
- **Ticket**: Ticket number.
- **Fare**: Passenger fare.
- **Cabin**: Cabin number.
- **Embarked**: Port of Embarkation; C = Cherbourg, Q = Queenstown, S = Southampton.

## Usage:

The Titanic dataset is often used for classification tasks to predict a passenger's survival based on their attributes. It is a great dataset for beginners to learn data preprocessing, feature engineering, and binary classification techniques.


In [ ]:
# Load Titanic dataset
titanic = sns.load_dataset('titanic')
titanic = titanic[['pclass', 'sex', 'age', 'fare', 'survived']].dropna()
X = titanic.drop('survived', axis=1)
y = titanic['survived']

In [ ]:
X.sample(5)

In [ ]:
X.info()

In [ ]:
X['pclass'] = X['pclass'].astype(str)

In [ ]:
# Convert categorical variables to dummy variables
X = pd.get_dummies(X, drop_first=True, dtype=int)

In [ ]:
X.sample(5)

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [ ]:
# Train a Random Forest classifier
classifier = RandomForestClassifier(random_state=42)
classifier.fit(X_train, y_train)

In [ ]:
# Evaluate the model
y_pred = classifier.predict(X_test)

# Calculate different evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
conf_matrix = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix:\n", conf_matrix)

In [ ]:
class_names = ['Died', 'Survived']  

conf_matrix_df = pd.DataFrame(conf_matrix,
                              index=class_names,
                              columns=class_names)


# Create the heatmap
plt.figure(figsize=(4, 4))
heatmap = sns.heatmap(conf_matrix_df,
                      annot=True,
                      fmt='d',
                      cmap='Blues',
                      linewidths=.5,
                      cbar=False)

# Add labels and title
plt.ylabel('True label', fontsize=12)
plt.xlabel('Predicted label', fontsize=12)
plt.title('Confusion Matrix')

plt.show()

In [ ]:
# Tell LIME which encoded columns are categorical, and keep arrays numeric.
categorical_indices = [X_train.columns.get_loc(name) for name in ['sex_male', 'pclass_2', 'pclass_3']]
explainer = LimeTabularExplainer(
    X_train.to_numpy(dtype=float),
    feature_names=X_train.columns.tolist(),
    class_names=['Died', 'Survived'],
    categorical_features=categorical_indices,
    discretize_continuous=True,
    random_state=42,
)

def predict_survival(rows):
    return classifier.predict_proba(pd.DataFrame(rows, columns=X_train.columns))

i = 23
exp = explainer.explain_instance(
    X_test.iloc[i].to_numpy(dtype=float), predict_survival,
    num_features=len(X.columns), labels=(1,),
)
print('Instance:', i)
print('True class:', y_test.iloc[i])
print('Predicted probabilities [Died, Survived]:', predict_survival(X_test.iloc[[i]])[0])
print('Contributions toward Survived:', exp.as_list(label=1))
from IPython.display import HTML
HTML(exp.as_html())

### Interpreting this explanation

The explanation above targets class 1 (`Survived`). A positive LIME weight increases the local surrogate's prediction for survival; a negative weight decreases it. Read the condition and sign together: a condition such as `sex_male=0` describes a different category from `sex_male=1`.

The model probabilities, true label, and feature contributions are printed from this run. LIME is a local approximation, so its fitted prediction can differ from the random forest's probability.

# Regression

In [ ]:
import pandas as pd
import numpy as np
import sklearn.ensemble
import lime
import lime.lime_tabular

In [ ]:
# Use a small regression dataset bundled with scikit-learn.
from sklearn.datasets import load_diabetes
diabetes = load_diabetes()
data, target = diabetes.data, diabetes.target

In [ ]:
feature_names = diabetes.feature_names

In [ ]:
rf = sklearn.ensemble.RandomForestRegressor(n_estimators=100, random_state=42)

In [ ]:
train, test, labels_train, labels_test = sklearn.model_selection.train_test_split(
    data, target, train_size=0.80, random_state=42)

In [ ]:
rf.fit(train, labels_train)

In [ ]:
print('Random Forest MSError', np.mean((rf.predict(test) - labels_test) ** 2))

In [ ]:
print('MSError when predicting the mean', np.mean((labels_train.mean() - labels_test) ** 2))

In [ ]:
categorical_features = np.argwhere(
    np.array([len(set(data[:, x]))
              for x in range(data.shape[1])]) <= 10).flatten()

In [ ]:
explainer = lime.lime_tabular.LimeTabularExplainer(
    train,
    feature_names=feature_names,
    class_names=['disease progression'],
    categorical_features=categorical_features,
    verbose=True,
    random_state=42,
    mode='regression')

In [ ]:
i = 25
exp = explainer.explain_instance(test[i], rf.predict, num_features=5)

In [ ]:
html = exp.as_html()
from IPython.display import HTML
HTML(html)

In [ ]:
exp.as_list()

In [ ]:
# recreating the prediction
sum([x[1] for x in exp.as_list()]) + exp.intercept[1]

The intercept plus the displayed feature contributions gives LIME's local surrogate prediction for this instance. This is an approximation to the random forest prediction, not an exact reconstruction of it.